<a href="https://colab.research.google.com/github/Hithavan31/BIS_PROGRAMS/blob/main/1BM24CS118_Solar_Panel_Placement_Optimization_Parallel_Cellular_Algorithm_Problem_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
from concurrent.futures import ThreadPoolExecutor
import random

# Calculate fitness of a panel configuration
def fitness(grid, sunlight):
    rows, cols = grid.shape
    total_sunlight = 0
    shadow_penalty = 0

    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                total_sunlight += sunlight[i][j]

                # Check neighbouring cells
                for x in range(max(0, i-1), min(rows, i+2)):
                    for y in range(max(0, j-1), min(cols, j+2)):
                        if (x, y) != (i, j) and grid[x][y] == 1:
                            shadow_penalty += 0.12 / 2

    return total_sunlight - shadow_penalty


# Generate random initial panel configuration
def initialize(sunlight, panel_count):
    grid = np.zeros(sunlight.shape, dtype=int)

    valid_cells = list(zip(*np.where(sunlight > 0)))
    positions = random.sample(valid_cells, panel_count)

    for i, j in positions:
        grid[i][j] = 1

    return grid


# Evaluate a possible movement
def evaluate_move(args):
    grid, sunlight, source, target = args

    new_grid = grid.copy()

    new_grid[source] = 0
    new_grid[target] = 1

    new_score = fitness(new_grid, sunlight)
    old_score = fitness(grid, sunlight)

    return new_score - old_score, source, target


# Parallel Cellular Algorithm
def parallel_cellular_algorithm(sunlight, panel_count):

    best_grid = None
    best_score = float('-inf')

    # Multiple initial configurations
    for restart in range(5):

        grid = initialize(sunlight, panel_count)

        for iteration in range(100):

            rows, cols = grid.shape
            occupied = list(zip(*np.where(grid == 1)))
            empty = list(zip(*np.where(
                (grid == 0) & (sunlight > 0)
            )))

            tasks = []

            # Generate neighbouring cell movements
            for source in occupied:
                for target in empty:

                    distance = max(
                        abs(source[0] - target[0]),
                        abs(source[1] - target[1])
                    )

                    if 0 < distance <= 2:
                        tasks.append(
                            (grid, sunlight, source, target)
                        )

            if not tasks:
                break

            # Evaluate all movements in parallel
            with ThreadPoolExecutor() as executor:
                results = list(executor.map(evaluate_move, tasks))

            # Sort movements by fitness improvement
            results.sort(reverse=True, key=lambda x: x[0])

            updated_grid = grid.copy()
            selected = []
            used_cells = set()

            # Resolve conflicts between parallel updates
            for improvement, source, target in results:

                if improvement <= 0:
                    continue

                affected = set()

                for cell in [source, target]:
                    for x in range(max(0, cell[0]-1),
                                   min(rows, cell[0]+2)):
                        for y in range(max(0, cell[1]-1),
                                       min(cols, cell[1]+2)):
                            affected.add((x, y))

                if affected.isdisjoint(used_cells):
                    selected.append((source, target))
                    used_cells.update(affected)

            if not selected:
                break

            # Apply simultaneous cell updates
            for source, target in selected:
                updated_grid[source] = 0
                updated_grid[target] = 1

            # Accept only if fitness improves
            if fitness(updated_grid, sunlight) > fitness(grid, sunlight):
                grid = updated_grid
            else:
                break

        current_score = fitness(grid, sunlight)

        if current_score > best_score:
            best_score = current_score
            best_grid = grid.copy()

    return best_grid, best_score


# ---------------- MAIN PROGRAM ----------------

print("SOLAR PANEL PLACEMENT OPTIMIZATION")
print("Using Parallel Cellular Algorithm")

rows = int(input("Enter rooftop rows: "))
cols = int(input("Enter rooftop columns: "))

sunlight = []

print("\nEnter sunlight exposure (0-10)")
print("Enter 0 for unavailable rooftop cells")

for i in range(rows):
    values = list(map(int, input(
        f"Enter sunlight values for row {i+1}: "
    ).split()))

    if len(values) != cols or any(v < 0 or v > 10 for v in values):
        raise ValueError("Enter exactly the required number of values between 0 and 10")

    sunlight.append(values)

sunlight = np.array(sunlight, dtype=float) / 10.0

panel_count = int(input("\nEnter number of solar panels: "))

if panel_count <= 0 or panel_count > np.count_nonzero(sunlight):
    raise ValueError("Invalid number of solar panels")

# Execute PCA
best_grid, best_score = parallel_cellular_algorithm(
    sunlight, panel_count
)

# Display optimized configuration
print("\nOPTIMIZED SOLAR PANEL CONFIGURATION")

for i in range(rows):
    for j in range(cols):
        if best_grid[i][j] == 1:
            print(" P ", end="")
        elif sunlight[i][j] == 0:
            print(" X ", end="")
        else:
            print(" . ", end="")
    print()

print("\nMaximum Fitness Score:", round(best_score, 4))
print("Number of Solar Panels:", np.sum(best_grid))
print("Total Sunlight Exposure:",
      round(np.sum(best_grid * sunlight), 4))


SOLAR PANEL PLACEMENT OPTIMIZATION
Using Parallel Cellular Algorithm
Enter rooftop rows: 4
Enter rooftop columns: 4

Enter sunlight exposure (0-10)
Enter 0 for unavailable rooftop cells
Enter sunlight values for row 1: 7 5 4 2
Enter sunlight values for row 2: 3 7 6 1
Enter sunlight values for row 3: 1 2 9 7 
Enter sunlight values for row 4: 3 5 6 9

Enter number of solar panels: 4

OPTIMIZED SOLAR PANEL CONFIGURATION
 P  .  .  . 
 .  .  P  . 
 .  .  P  . 
 .  .  .  P 

Maximum Fitness Score: 2.86
Number of Solar Panels: 4
Total Sunlight Exposure: 3.1
